# 🫀 퀘스트 46 · Q7-S″ — **재계산**: 프레임을 고치고 영점을 기준으로 삼는다

| | **MedKOS / `notebooks/quest46_q7s3_recompute.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` |
| 앞선 실험 | **Q7-S′**(`quest46_q7s2_p_aligned` · 공식 실행 `20260804T0658`) |
| 규약 | **R11 · R16 · R22 · R24 · R29 ② · R30 ① · R33 ① · R34 ②③ · R35 ④⑦ · R36 ①②④⑤** |
| 학습 | 로지스틱 회귀만 · GPU 불필요 · **새 데이터 0** · 예상 25~40분 |

## 왜 다시 도나 — 새 데이터 없이 판정이 바뀔 수 있다

Q7-S′ 는 **자료가 아니라 프레임**에서 하나를 틀렸다. 외부 검토가 잡아냈다.

### ★★★ 틀린 것: 필요 표본을 **등가 프레임**으로 계산했다

Q7-S′ 의 `need_n()` 은 이 식이다.

```
n_req = n × (CI반폭 / (여유 0.01 − |점추정|))²
```

이건 **「P 가 아무것도 안 준다」를 증명**하는 데 필요한 수다(등가). 그래서
`S3` 는 초과 +0.0272 가 여유 0.01 **밖**이라 분모가 음수가 되고 **「도달 불가」**
로 찍혔다. **신호가 약해서가 아니라, 등가를 선언할 수 없어서다.**

우리가 알고 싶은 건 정반대다 — **우월**(효과가 0 이 아님). 그 식은

```
n_req = n × (CI반폭 / 점추정)²          (CI 가 0 을 배제 · 검정력 50%)
      × 2.04                            (검정력 80%)
```

| 관문 | 점추정 | 반폭 | **등가**(옛) | **우월 50%** | **우월 80%** |
|---|---:|---:|---:|---:|---:|
| S1 | +0.0029 | 0.0141 | 222 | 1,324 | 2,701 |
| S2 | +0.0035 | 0.0143 | 269 | 935 | 1,907 |
| S5 | +0.0134 | 0.0224 | — | 156 | 318 |
| **S3** | **+0.0272** | 0.0531 | **도달 불가** | **130 매칭가능 → 총 214** | **437** |

**S3 는 네 관문 중 가장 싼 관문이다.** 「도달 불가」는 프레임의 산물이었다.
이 셀들이 그 표를 **자료에서 다시 계산**해 박는다.

⚠️ 다만 결론이 통째로 뒤집히진 않는다 — 80% 검정력이면 S3 437 · S5 318 이고,
**SVDB 풀 126 은 물론 풀링 코호트 201 로도 못 넘는다.** 바뀌는 건
**「불가능」 → 「비싸다」**이지 「도달권」이 아니다. 그 사실도 같이 찍는다.

### ★★ 고칠 것 둘

**① 주 대비가 빈 것끼리 비교였다.** S5 는 `palign − pmorph` 인데 `pmorph`(−0.0100)가
자기 영점 `shuf5`(−0.0013 · MDE 0.0107)의 CI 안에 통째로 들어간다 — **`pmorph` 도
비어 있다.** 빈 것에서 빈 것을 뺀 차이는 창 특이성을 증명하지 못한다.
→ **주 대비를 `palign − shuf5`(짝지은 차)로 승격**한다. 같은 폴드·같은 레코드라
공통 분산이 상쇄돼 CI 도 좁아진다.

**② 「정확 매칭」이 정말 정확한지 안 쟀다.** S3 의 매칭 키는 `np.round(f1)` 이라
**한 층 안에서 `f1` 이 최대 1샘플 흔들린다.** 그 잔여로 조기성이 새면 S3 는
「리듬이 아니다」를 보증하지 못한다. → **`f1` 자신을 같은 매칭에 통과**시켜
**항등 대조**로 쓴다(R35 ④). 0.5 에서 떨어지면 매칭이 새는 것이다.

## 사전등록 — 관문

| 관문 | 내용 | 판정 |
|---|---|---|
| **W0** | **재현 증명** — Q7-S′ 공식 실행값을 소수점까지 다시 낸다 | 어긋나면 **중단**(R35 ⑦) |
| **W1 ★★★(주)** | **`palign − shuf5` 짝지은 차** — 창 특이성의 올바른 대비 | CI 하한 > 0 |
| **W2** | 필요 표본을 **우월 프레임**으로 다시 계산(50%·80% 병기) | 계산 · 판정 아님 |
| **W3** | 추론 단위 민감도 — 레코드 부트 vs 쌍 풀링 | MDE 0.0531 이 진짜인가 |
| **W4** | **전 팔을 자기 영점과 짝지어** 재보고(k별) | 영점 드리프트가 실재하나 |
| **W5 ★★** | **매칭 폭 곡선 + `f1` 항등 대조** | w=1 에서 `f1` ≈ 0.5 여야 |

### 판정표 (R29 ②)

- **W0 실패** → 자산·환경이 바뀐 것이다. **어떤 관문도 읽지 않는다**
- **W5 에서 `f1` 이 0.5 를 뗀다** → S3 는 조기성을 통제한 적이 없다. **S3 를 철회**한다
- **W1 ✅** → 창 내용이 특이적이다. S3 의 +0.0272 를 그대로 읽어도 된다
- **W1 ⚠️ · W5 정상** → 신호 크기가 문제다. 다음은 **Q7-V**(자가 병목인지)

⚠️ **이 런은 딥러닝 진입 조건을 바꾸지 않는다.** 프레임을 고칠 뿐이다.

In [ ]:
# CELL 0 — 공용 사전점검 (Q7-S′ 와 **동일** — 재현 증명을 위해 손대지 않는다)
import numpy as np

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def mde(lo, hi):
    """최소 검출 효과 = CI 반폭(R33 ①)."""
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def need_equiv(n, lo, hi, mean, margin):
    """★ **등가** 필요 표본 — Q7-S′ 가 쓴 식. 비교용으로 남긴다."""
    if not (np.isfinite(lo) and np.isfinite(hi) and np.isfinite(mean)) or n < 1:
        return float("nan")
    slack = margin - abs(mean)
    return None if slack <= 0 else float(n) * (((hi - lo) / 2.0) / slack) ** 2

def need_super(n, lo, hi, mean, power80=False):
    """★★ **우월** 필요 표본 — CI 가 0(또는 기준)을 배제하려면.

    반폭 ∝ 1/√n 이므로  n_req = n × (반폭/|효과|)².
    ★ 이건 **검정력 50%** 기준이다(관측 효과가 참값이고 CI 가 딱 0 에 닿는 지점).
      관례적 80% 검정력은 ×(1.96+0.84)²/1.96² = **×2.04**."""
    if not (np.isfinite(lo) and np.isfinite(hi) and np.isfinite(mean)) or n < 1:
        return float("nan")
    if abs(mean) < 1e-12:
        return float("inf")
    r = float(n) * (((hi - lo) / 2.0) / abs(mean)) ** 2
    return r * 2.04 if power80 else r

def boot_mean(v, seed, nb=3000, q=2.5):
    """**레코드 단위** 부트스트랩(R11 — 환자가 단위다)."""
    d = np.asarray(v, float); d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), len(d)
    rng = np.random.RandomState(seed)
    b = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)]
    return (float(d.mean()), float(np.percentile(b, q)),
            float(np.percentile(b, 100 - q)), len(d))

class AssetError(RuntimeError): pass
print("CELL 0 ✅")

In [ ]:
# CELL 1 — 설정 · 사전등록 · ★ Q7-S′ 공식 실행값(재현 증명 기준)
import os, sys, json, importlib, time, warnings
importlib.invalidate_caches(); warnings.filterwarnings("ignore")
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

# ★★ Q7-S′ 와 **완전히 같은** 상수 — 하나라도 다르면 재현 증명이 깨진다
SEED0, IDX_S = 20260804, 1
FS, RPRE, L = 360, 100, 300
QUANT_MS = 1000.0 / 128.0
FULL_K = tuple(range(4, 33))
RHY_K  = (5, 10, 20, 32)
assert set(RHY_K) <= set(FULL_K)
GATE_ARM = "palign"
ARMS = ("palign", "pmorph", "noise5", "shuf5")
N_DIM = 5
PMORPH_WIN = (31, 53)
K_PERS = (5, 10, 20)
SPEC_LO = 0.95
MIN_S, MIN_N = 25, 25
MIN_PAIR = 200
N_SHUF = 20

# ── ★ 새로 들어온 것
ZERO_ARM   = "shuf5"                 # ★★ W1 — 주 대비의 기준을 **영점**으로
MATCH_W    = (1, 2, 4, 8, 16)        # ★★ W5 — 매칭 폭(샘플) 곡선. 1 = Q7-S′ 의 「정확」
IDENTITY_TOL = 0.02                  # W5 항등 대조 허용(f1 이 0.5±이 안이어야)
POOL_NOW, POOL_MAX = 126, 201        # 도달 가능 풀(SVDB78+MITBIH48) · 풀링 상한

SV5   = os.path.join(MITBIH, "svdb_data5.npz")
PDEL  = os.path.join(MITBIH, "svdb_pdelin.npz")

# ★★★ Q7-S′ 공식 실행 `20260804T0658` 의 출력값 — **재현 증명의 기준**
#     로그에 소수점 4자리로 찍혔으므로 반올림 오차 5e-5 는 구조적으로 존재한다.
REF = {
    "S2.palign": +0.0035, "S2.pmorph": -0.0100,
    "S2.noise5": -0.0002, "S2.shuf5":  -0.0013,
    "S1.palign@k5": +0.0036, "S1.palign@k10": +0.0034, "S1.palign@k20": +0.0030,
    "S3.p_score": 0.5362, "S3.pr_q8": 0.4974, "S3.p_miss": 0.4801,
    "S5.LORO": +0.0134,
}
REF_N = {"S2": 56, "S3": 34}
TOL_WARN, TOL_FAIL = 1e-3, 5e-3      # 경고 / 중단

RULE_CHECK = {
    "R11 환자 단위":      "부트스트랩·판정 전부 **레코드 단위**",
    "R16 fallback 없음":  "자산 없으면 **중단**",
    "R22 누수 없음":      "표준화·PCA·개인화는 **학습 비트에서만**",
    "R29 ② 분기 금지":    "W0 이 깨지면 어떤 관문도 읽지 않는다",
    "R30 ① 필요표본":     "★ **등가·우월 두 프레임을 나란히** 찍는다",
    "R33 ① MDE":          "관문마다 MDE 를 내고 점추정과 비교",
    "R34 ② 선택 편의":    "★ 프로브 선택도 **LORO** 로 — Q7-S′ 는 max 로 골랐다(자백)",
    "R34 ③ 대조 보장":    "★★ **주 대비를 `palign − shuf5` 로** — 영점이 기준이다",
    "R35 ④ 항등 대조":    "★★ **`f1` 자신을 매칭에 통과** — 0.5 에서 떨어지면 매칭이 샌다",
    "R35 ⑦ 정합 증명":    "★★ W0 — Q7-S′ 값을 다시 내지 못하면 **중단**",
    "R36 ① 상한":         "미결이면 CI 상단을 **상한**으로 명시한다",
    "R36 ⑤ 성분 병기":    "★ 차이(A−B)는 **성분과 함께만** 인용한다",
}

CONFIG = dict(
    exp="quest46_q7s3_recompute", quest="ailab-2026-0046", step="recompute-frames",
    parent_exp=["quest46_q7s2_p_aligned"],
    purpose=("Q7-S′ 를 **같은 자료로 다시 세운다**(새 데이터 0). 외부 검토가 프레임 오류 "
             "하나를 잡았다 — 필요 표본을 **등가**로 계산해서 초과 +0.0272 인 S3 를 "
             "「도달 불가」로 찍었다. 우월 프레임에서 S3 는 **가장 싼 관문**이다. "
             "동시에 주 대비를 `palign − pmorph`(빈 것끼리)에서 **`palign − shuf5`"
             "(영점 기준)** 로 옮기고, 「정확 매칭」이 정말 정확한지 **`f1` 항등 대조**로 "
             "확인한다"),
    dataset="SVDB 78레코드 184,499비트(svdb_data5.npz) + P 위치 표(svdb_pdelin.npz)",
    arms=list(ARMS), gate_arm=GATE_ARM, zero_arm=ZERO_ARM, n_dim=N_DIM,
    k_pers=list(K_PERS), quant_ms=QUANT_MS, full_k=list(FULL_K), rhy_k=list(RHY_K),
    match_w=list(MATCH_W), pool_now=POOL_NOW, pool_max=POOL_MAX,
    ref=REF, ref_n=REF_N, tol=dict(warn=TOL_WARN, fail=TOL_FAIL),
    rule_check=RULE_CHECK,
    predictions={
        "W0": "**재현 증명** — Q7-S′ 공식 실행 `20260804T0658` 의 S1·S2·S3·S5 를 다시 낸다. "
              f"|Δ| > {TOL_FAIL} 이면 **중단**(자산이나 환경이 바뀐 것이고, 그러면 아래 "
              "관문의 해석 기준이 사라진다)",
        "W1": "★★★ **(주) `palign − shuf5` 짝지은 차** — Q7-S′ 의 S5 는 `palign − pmorph` "
              "인데 `pmorph`(−0.0100)가 자기 영점 `shuf5`(−0.0013 · MDE 0.0107)의 CI 안에 "
              "들어간다. **빈 것에서 빈 것을 뺀 차이**라 창 특이성을 증명하지 못한다. "
              "영점을 기준으로 삼으면 그게 곧 **창 내용의 특이성**이다. CI 하한 > 0",
        "W2": "필요 표본을 **우월 프레임**으로 다시. 등가값과 나란히 찍어 프레임이 판정을 "
              "얼마나 바꾸는지 보인다. 검정력 50%·80% 둘 다. **판정 아님**",
        "W3": "추론 단위 민감도 — ⓐ 레코드평균+레코드부트(현행) ⓑ 쌍풀링+레코드부트 "
              "ⓒ 쌍풀링+쌍부트(**군집 무시 · 추론에 쓰면 안 됨** · 군집 비용 정량용). "
              "ⓐ≈ⓑ 면 MDE 0.0531 은 진짜다",
        "W4": "**전 팔을 자기 영점과 짝지어** k별 재보고. 외부 검토가 `shuf5` 의 k=20 "
              "드리프트(−0.0066)를 지적했는데, 그게 `shuf5` 자기 MDE(0.0107) **안**인지 "
              "밖인지부터 가른다. 안이면 드리프트는 확인된 게 아니다",
        "W5": "★★ **매칭 폭 곡선 + `f1` 항등 대조.** 매칭 키가 `np.round(f1)` 이라 한 층 "
              "안에서 `f1` 이 최대 1샘플 흔들린다. **`f1` 자신을 같은 매칭에 통과**시켜 "
              f"0.5±{IDENTITY_TOL} 안이어야 「조기성을 통제했다」가 성립한다. 폭을 넓히면 "
              "레코드는 늘지만 조기성이 새므로, **누출을 잰 뒤에만** 넓힌 값을 쓴다"},
    caveat=("★ **새 데이터가 0 이다** — 같은 자산, 같은 코드, 같은 시드. 바뀌는 건 "
            "**해석 프레임과 대비 상대**뿐이다. "
            "★ **딥러닝 진입 조건을 바꾸지 않는다** — 그건 Q7-V 와 그 다음 몫이다. "
            "★ W5 가 실패하면 **S3 를 철회**해야 한다 — 가장 아픈 가능성이고, 그래서 "
            "먼저 잰다. "
            "★ 로지스틱 회귀만 · 예상 25~40분"))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q7s3_recompute", CONFIG, project=PROJECT)
run.log("설정 ✅ **Q7-S″ 재계산** — 새 데이터 0 · 프레임과 대비 상대만 바꾼다")
run.log(f"  ★★ 주 대비를 `{GATE_ARM} − {ZERO_ARM}` 로 승격 — 영점이 기준이다(R34 ③)")
run.log(f"  ★★ 매칭 폭 곡선 {MATCH_W} 샘플 + **`f1` 항등 대조**(R35 ④)")
run.log(f"  ★  필요 표본을 **등가·우월** 두 프레임으로 병기(R30 ①)")
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<18} {v_}")

In [ ]:
# CELL 2 — 자산 · 정합 재확인 (Q7-S′ CELL 2 와 **동일**)
run.log("\n" + "=" * 100)
run.log("【S3-0】 자산 적재 + 좌표 정합 재확인")
run.log("=" * 100)
for p_, why in ((SV5, "svdb_labels.py build"), (PDEL, "**Q7-P0 를 먼저 돌린다**")):
    if not os.path.exists(p_):
        raise AssetError(f"{p_} 없음 — {why}(R16)")
D5 = np.load(SV5, allow_pickle=True)
PD = np.load(PDEL, allow_pickle=True)
PID = np.asarray(D5["pid"]).astype(int)
SYM = np.asarray(D5["sym"]).astype(str)
Y3  = np.asarray(D5["y3"]).astype(int)
PRE = np.asarray(D5["pre_rr"], float)
POST = np.asarray(D5["post_rr"], float)
BEAT = np.asarray(D5["beat"])
for f in ("p_idx", "p_score", "pid", "sym"):
    if f not in PD.files:
        raise AssetError(f"P 위치 표에 `{f}` 가 없다 — 보유 {sorted(PD.files)}")
if len(PD["p_idx"]) != len(PID):
    raise AssetError(f"길이 불일치 — pdelin {len(PD['p_idx']):,} vs d5 {len(PID):,}")
_bp = int((np.asarray(PD["pid"]).astype(int) != PID).sum())
_bs = int((np.asarray(PD["sym"]).astype(str) != SYM).sum())
if _bp or _bs:
    raise AssetError(f"정합 깨짐 — pid {_bp}개 · sym {_bs}개. Q7-P0 를 다시 돌린다")
run.log(f"  ✅ 정합 재확인 — {len(PID):,} 비트 전부 일치(pid·sym)")
P_IDX = np.asarray(PD["p_idx"]).astype(int)
P_SC  = np.asarray(PD["p_score"], float)
KEEP = Y3 >= 0
run.log(f"  비트 {len(PID):,} → 유효 {int(KEEP.sum()):,} · 레코드 {len(np.unique(PID))}")
CONFIG["asset"] = dict(n=int(len(PID)), n_keep=int(KEEP.sum()),
                       n_rec=int(len(np.unique(PID))), align_ok=True)
run.save_json("config", CONFIG)

In [ ]:
# CELL 3 — 특징 (Q7-S′ CELL 3 과 **바이트 단위로 같은 논리** — 재현 증명 전제)
from sklearn.decomposition import PCA
import pandas as pd
run.log("\n" + "=" * 100)
run.log("【S3-A】 특징 — Q7-S′ 와 동일하게 재구성")
run.log("=" * 100)
K = np.where(KEEP)[0]
RID = PID[K]; Y = Y3[K]; TT = (Y == IDX_S)
pre = PRE[K].astype(float); post = POST[K].astype(float)
pidx = P_IDX[K]; psc = P_SC[K]
RS = np.array(sorted(set(RID.tolist())))
_S = pd.Series(pre); _G = _S.groupby(pd.Series(RID))

def local_base(k):
    r = _G.apply(lambda x: x.shift(1).rolling(k, min_periods=1).median())
    r = np.asarray(r).astype(float)
    return np.where(np.isfinite(r), r, pre)

_med = _G.transform("median").to_numpy()
_std = _G.transform("std").to_numpy()
_mean = _G.transform("mean").to_numpy()
f1 = _med - pre
f2 = {k: 1.0 - pre / (local_base(k) + 1e-9) for k in FULL_K}
f3 = post - pre
f4 = np.nan_to_num(_std / (_mean + 1e-9))
RHY = np.c_[f1, np.column_stack([f2[k] for k in RHY_K]), f3, f4,
            np.log1p(pre), np.log1p(post)]

MISS = pidx < 0
pr_ms = np.where(MISS, np.nan, (RPRE - pidx) / FS * 1000.0)
pr_q8 = np.round(pr_ms / QUANT_MS) * QUANT_MS
nmask = (Y == 0)
pr_ref = np.full(len(pr_q8), np.nan); sc_ref = np.full(len(psc), np.nan)
sc_mad = np.full(len(psc), np.nan)
for u in np.unique(RID):
    m = RID == u; mn = m & nmask
    a = pr_q8[mn][np.isfinite(pr_q8[mn])]; b = psc[mn]
    pr_ref[m] = np.median(a) if len(a) >= 5 else np.nan
    sc_ref[m] = np.median(b) if len(b) >= 5 else np.nan
    sc_mad[m] = (np.median(np.abs(b - np.median(b))) + 1e-9) if len(b) >= 5 else np.nan
pr_dev = pr_q8 - pr_ref
sc_dev = (psc - sc_ref) / sc_mad
PAL = np.c_[psc, np.nan_to_num(pr_q8, nan=0.0), np.nan_to_num(pr_dev, nan=0.0),
            np.nan_to_num(sc_dev, nan=0.0), MISS.astype(float)]
if PAL.shape[1] != N_DIM:
    raise AssetError(f"P 정렬 특징이 {PAL.shape[1]}차원 — {N_DIM} 이어야 한다")
Bk = np.ascontiguousarray(BEAT[K]).astype("float32")
PM_RAW = Bk[:, 0, PMORPH_WIN[0]:PMORPH_WIN[1]].astype(float)
_rng = np.random.RandomState(SEED0 + 99)
NOISE = _rng.normal(size=(len(K), N_DIM))
run.log(f"  리듬 기저 {RHY.shape[1]}차원 · P 정렬 {PAL.shape[1]}차원 · "
        f"결측 {MISS.mean():.4f} · 레코드 {len(RS)}")
run.log(f"  ★ `f1` 단위 = **샘플**(360Hz) — 매칭 폭 1샘플 = {1000/FS:.2f}ms")
CONFIG["feat"] = dict(rhy_dim=int(RHY.shape[1]), pal_dim=int(PAL.shape[1]),
                      miss_rate=float(MISS.mean()), f1_unit_ms=float(1000.0 / FS))
run.save_json("config", CONFIG)

In [ ]:
# CELL 4 — LORO 루프 (Q7-S′ CELL 4 와 **동일**) — 재현 증명의 재료
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_curve
run.log("\n" + "=" * 100)
run.log("【S3-B】 LORO 루프 재실행 — Q7-S′ 와 같은 시드·같은 순서")
run.log("=" * 100)
VERD, DIFF = {}, {}
def g_(k, v, d):
    VERD[k] = v; run.log(f"  {k:<4}{v}  {d}")

def partial_auc(y, s, spec_lo):
    fpr, tpr, _ = roc_curve(y, s)
    hi = 1.0 - spec_lo
    m = fpr <= hi
    if m.sum() < 2:
        return float("nan")
    x, yv = fpr[m], tpr[m]
    return float((np.diff(x) * (yv[:-1] + yv[1:]) / 2.0).sum()) / hi

def fit_eval(Xtr, ytr, Xte, yte):
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-9
    lr = LogisticRegression(max_iter=3000, C=1.0)
    lr.fit((Xtr - mu) / sd, ytr)
    s = lr.decision_function((Xte - mu) / sd)
    return average_precision_score(yte, s), partial_auc(yte, s, SPEC_LO)

def feats_for(arm, tr, te):
    if arm == "noise5":
        return NOISE[tr], NOISE[te]
    if arm == "pmorph":
        pca = PCA(n_components=N_DIM, random_state=SEED0).fit(PM_RAW[tr])
        return pca.transform(PM_RAW[tr]), pca.transform(PM_RAW[te])
    Ztr, Zte = PAL[tr].copy(), PAL[te].copy()
    if arm == "shuf5":
        rr = np.random.RandomState(SEED0 + 7)
        for Z_, m_ in ((Ztr, tr), (Zte, te)):
            for u in np.unique(RID[m_]):
                sel = np.where(RID[m_] == u)[0]
                Z_[sel] = Z_[sel][rr.permutation(len(sel))]
    return Ztr, Zte

D_LORO = {a: np.full(len(RS), np.nan) for a in ARMS}
D_PERS = {k: {a: np.full(len(RS), np.nan) for a in ARMS} for k in K_PERS}
T0 = time.time()
for i, r in enumerate(RS):
    te0 = np.where(RID == r)[0]
    s_idx = te0[TT[te0]]; n_idx = te0[~TT[te0]]
    if len(s_idx) < MIN_S or len(n_idx) < MIN_N:
        continue
    rng = np.random.RandomState(SEED0 + 313 + int(r))
    pick = {k: (rng.choice(s_idx, k, replace=False), rng.choice(n_idx, k, replace=False))
            for k in K_PERS if len(s_idx) > k and len(n_idx) > k}
    if not pick:
        continue
    used = np.unique(np.concatenate([np.r_[a, b] for a, b in pick.values()]))
    ev = np.setdiff1d(te0, used)
    tr0 = np.where(RID != r)[0]
    if len(ev) < 30 or TT[ev].sum() < 3:
        continue
    base_i = fit_eval(RHY[tr0], TT[tr0].astype(int), RHY[ev], TT[ev].astype(int))[0]
    base_p = {}
    for k in pick:
        tk = np.r_[tr0, pick[k][0], pick[k][1]]
        base_p[k] = fit_eval(RHY[tk], TT[tk].astype(int), RHY[ev], TT[ev].astype(int))[0]
    for a in ARMS:
        Ztr, Zev = feats_for(a, tr0, ev)
        D_LORO[a][i] = fit_eval(np.c_[RHY[tr0], Ztr], TT[tr0].astype(int),
                                np.c_[RHY[ev], Zev], TT[ev].astype(int))[0] - base_i
        for k in pick:
            tk = np.r_[tr0, pick[k][0], pick[k][1]]
            Zk, Zev2 = feats_for(a, tk, ev)
            D_PERS[k][a][i] = fit_eval(np.c_[RHY[tk], Zk], TT[tk].astype(int),
                                       np.c_[RHY[ev], Zev2], TT[ev].astype(int))[0] - base_p[k]
    if (i + 1) % 10 == 0:
        run.log(f"    {i+1}/{len(RS)}  ({time.time()-T0:.0f}초)")
S2 = {}
for a in ARMS:
    m_, lo_, hi_, n_ = boot_mean(D_LORO[a], SEED0 + 21)
    S2[a] = dict(mean=m_, lo=lo_, hi=hi_, n=n_, mde=float(mde(lo_, hi_)))
S1 = {}
for k in K_PERS:
    for a in ARMS:
        m_, lo_, hi_, n_ = boot_mean(D_PERS[k][a], SEED0 + 31 + k)
        S1[f"{a}@k{k}"] = dict(mean=m_, lo=lo_, hi=hi_, n=n_, mde=float(mde(lo_, hi_)))
run.log(f"  ({time.time()-T0:.0f}초) 판정 레코드 {S2[GATE_ARM]['n']}")
CONFIG["S1"] = S1; CONFIG["S2"] = S2
run.save_json("config", CONFIG)

In [ ]:
# CELL 5 — 【S3-C】 S3 재계산 + ★★ W5 매칭 폭 곡선 · **`f1` 항등 대조**
run.log("\n" + "=" * 100)
run.log("【S3-C】 S3 재계산 · W5 매칭 폭 곡선 · **`f1` 항등 대조**(R35 ④)")
run.log("=" * 100)
run.log("  ▸ Q7-S′ 의 매칭 키는 `np.round(f1)` 이다 — 한 층 안에서 `f1` 이 **최대 1샘플**")
run.log("    흔들린다. 그 잔여로 조기성이 새면 S3 는 「리듬이 아니다」를 보증하지 못한다")
run.log("  ▸ 그래서 **`f1` 자신**을 같은 매칭에 통과시킨다. 0.5 를 떼면 매칭이 새는 것이다")

def basis_ext(idx):
    return np.c_[np.ones(len(idx)), f1[idx], f3[idx], f4[idx],
                 np.column_stack([f2[k][idx] for k in FULL_K])]

def resid(v, idx):
    X = basis_ext(idx); y = v[idx]
    ok = np.isfinite(y)
    if ok.sum() < X.shape[1] + 5:
        return np.full(len(idx), np.nan)
    b = np.linalg.lstsq(X[ok], y[ok], rcond=None)[0]
    return y - X @ b

def match_key(idx, w):
    """폭 `w` 샘플의 매칭 키. w=1 이 Q7-S′ 의 「정확 매칭」(np.round 와 동치)."""
    return np.floor(f1[idx] / float(w) + 0.5).astype(int) if w > 1 else \
           np.round(f1[idx]).astype(int)

def matched_stats(vsub, idx, w=1, shuffle_seed=None):
    """(AUROC, 쌍 수) — 같은 레코드·같은 `f1` 층 안 Mann-Whitney."""
    tt = TT[idx]
    if shuffle_seed is not None:
        rr = np.random.RandomState(shuffle_seed); tt = tt.copy()
        for u in np.unique(RID[idx]):
            m = np.where(RID[idx] == u)[0]
            tt[m] = tt[m][rr.permutation(len(m))]
    key = match_key(idx, w)
    win = tie = tot = 0.0
    for kk in np.unique(key):
        m = np.where(key == kk)[0]
        a = vsub[m[tt[m]]]; b = vsub[m[~tt[m]]]
        a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
        if not len(a) or not len(b):
            continue
        d = a[:, None] - b[None, :]
        win += float((d > 0).sum()); tie += float((d == 0).sum()); tot += float(d.size)
    return ((win + 0.5 * tie) / tot, int(tot)) if tot >= MIN_PAIR else (float("nan"), int(tot))

PROBES = {"p_score": psc, "pr_dev": pr_dev, "sc_dev": sc_dev,
          "pr_q8": pr_q8, "p_miss": MISS.astype(float)}
REC_OK = [r for r in RS
          if (lambda i: TT[i].sum() >= MIN_S and (~TT[i]).sum() >= MIN_N)(np.where(RID == r)[0])]
run.log(f"\n  판정 후보 레코드 {len(REC_OK)}")

# ── S3 재계산 (w=1 · Q7-S′ 와 동일) — **쌍 단위 합계도 함께 저장**(W3 용)
S3, PAIRW = {}, {}
for nm, v in PROBES.items():
    per, nul, npair, pw = [], [], [], []
    for r in REC_OK:
        idx = np.where(RID == r)[0]
        vr = resid(v, idx)
        a, npr = matched_stats(vr, idx, 1)
        if np.isfinite(a):
            per.append(a); npair.append(npr); pw.append((r, a, npr))
            nul.append(np.nanmean([matched_stats(vr, idx, 1, SEED0 + 700 + s)[0]
                                   for s in range(3)]))
    m_, lo_, hi_, n_ = boot_mean(per, SEED0 + 51)
    nm_, nlo, nhi, _ = boot_mean(nul, SEED0 + 52)
    S3[nm] = dict(auc=m_, lo=lo_, hi=hi_, n=n_, null=nm_, null_lo=nlo, null_hi=nhi,
                  mde=float(mde(lo_, hi_)),
                  pairs=int(np.median(npair)) if npair else 0)
    PAIRW[nm] = pw
    run.log(f"    {nm:<9} AUROC **{m_:.4f}** [{lo_:.4f}, {hi_:.4f}] · null {nm_:.4f} · "
            f"n={n_} · 쌍 중앙 {S3[nm]['pairs']:,} · MDE {mde(lo_,hi_):.4f}")

# ── ★★ W5 — 매칭 폭 곡선 + `f1` 항등 대조
run.log("\n  ★★ W5 — 매칭 폭 곡선 (`f1` 은 **잔차화하지 않은 원값**으로 통과 = 항등 대조)")
run.log(f"    {'w(샘플)':<9}{'ms':>7}{'레코드':>8}{'f1 항등':>10}{'p_score':>10}{'누출':>9}")
W5 = {}
for w in MATCH_W:
    fa, pa, nrec = [], [], 0
    for r in REC_OK:
        idx = np.where(RID == r)[0]
        a_f1, np1 = matched_stats(f1[idx], idx, w)          # ★ 원 `f1` — 항등 대조
        a_ps, _ = matched_stats(resid(psc, idx), idx, w)
        if np.isfinite(a_f1):
            fa.append(a_f1); nrec += 1
        if np.isfinite(a_ps):
            pa.append(a_ps)
    fm, flo, fhi, fn = boot_mean(fa, SEED0 + 81 + w)
    pm, plo, phi, pn = boot_mean(pa, SEED0 + 91 + w)
    W5[w] = dict(f1_auc=fm, f1_lo=flo, f1_hi=fhi, n_rec=int(fn),
                 ps_auc=pm, ps_lo=plo, ps_hi=phi, ps_n=int(pn),
                 leak=float(abs(fm - 0.5)) if np.isfinite(fm) else float("nan"))
    run.log(f"    {w:<9}{w*1000.0/FS:>7.1f}{fn:>8}{fm:>10.4f}{pm:>10.4f}"
            f"{abs(fm-0.5):>9.4f}")
w1 = W5.get(1, {})
if not np.isfinite(w1.get("f1_auc", np.nan)):
    g_("W5", "⛔ 측정 불가", "w=1 에서 `f1` 항등 대조가 안 섰다")
else:
    ok5 = abs(w1["f1_auc"] - 0.5) <= IDENTITY_TOL
    g_("W5", "✅ 지지" if ok5 else "❌ 기각",
       f"★★ `f1` 항등 대조 @w=1 **{w1['f1_auc']:.4f}** [{w1['f1_lo']:.4f}, "
       f"{w1['f1_hi']:.4f}] · 허용 0.5±{IDENTITY_TOL} · 누출 {w1['leak']:.4f}")
    if not ok5:
        run.log("       ⛔ **매칭이 샌다 — S3 는 조기성을 통제한 적이 없다.** S3 를 철회한다")
    else:
        run.log("       ▸ 매칭 안에서 조기성은 실제로 상수다. S3 의 초과분을 그대로 읽어도 된다")

# ── ★ 프로브 선택을 **LORO 로**(Q7-S′ 는 max 로 골랐다 — 자백한 편의 · R34 ②)
PN = [p for p in PROBES if np.isfinite(S3[p]["auc"])]
if not PN:
    g_("S3", "⛔ 측정 불가", "층이 선 레코드가 없다")
else:
    best_max = max(PN, key=lambda p: S3[p]["auc"])
    per_by_probe = {p: {r: a for r, a, _ in PAIRW[p]} for p in PN}
    sel, sel_names = [], []
    for r in REC_OK:
        cand = [p for p in PN if r in per_by_probe[p]]
        if not cand:
            continue
        bp = max(cand, key=lambda p: np.nanmean(
            [per_by_probe[p][o] for o in per_by_probe[p] if o != r] or [-np.inf]))
        sel.append(per_by_probe[bp][r]); sel_names.append(bp)
    ml, ll, hl, nl = boot_mean(sel, SEED0 + 55)
    S3["__loro__"] = dict(auc=ml, lo=ll, hi=hl, n=nl, mde=float(mde(ll, hl)),
                          picked={p: sel_names.count(p) for p in set(sel_names)},
                          max_probe=best_max, max_auc=S3[best_max]["auc"])
    bias = S3[best_max]["auc"] - ml
    run.log(f"\n  ★ 프로브 선택 편의 정량 — max 선택 `{best_max}` {S3[best_max]['auc']:.4f} vs "
            f"**LORO 선택 {ml:.4f}** [{ll:.4f}, {hl:.4f}] → 편의 **{bias:+.4f}**")
    run.log(f"    LORO 가 고른 프로브 분포 {S3['__loro__']['picked']}")
    DIFF["S3"] = dict(probe="LORO", **{k: v for k, v in S3["__loro__"].items()
                                       if k in ("auc", "lo", "hi", "n", "mde")},
                      null=S3[best_max]["null"])
    g_("S3", decide(ll, hl, 0.5, ">"),
       f"★★★ **LORO 프로브 선택** AUROC **{ml:.4f}** [{ll:.4f}, {hl:.4f}] · "
       f"null {S3[best_max]['null']:.4f} · n={nl} (max 선택이면 {S3[best_max]['auc']:.4f})")
CONFIG["S3"] = {k: v for k, v in S3.items()}; CONFIG["W5"] = W5
run.save_json("config", CONFIG)

In [ ]:
# CELL 6 — 【S3-D】 ★★★ W1 주 대비 `palign − shuf5` · W4 영점 짝지은 재보고
run.log("\n" + "=" * 100)
run.log("【S3-D】 W1(주) — `palign − shuf5` 짝지은 차 · W4 — 전 팔을 자기 영점과 짝지어")
run.log("=" * 100)
run.log("  ▸ Q7-S′ 의 S5 는 `palign − pmorph` 였는데 **`pmorph` 도 비어 있다** —")
run.log(f"    pmorph {S2['pmorph']['mean']:+.4f} 가 shuf5 CI "
        f"[{S2['shuf5']['lo']:+.4f}, {S2['shuf5']['hi']:+.4f}] 안에 들어간다")
run.log("  ▸ 빈 것에서 빈 것을 뺀 차이는 **창 특이성을 증명하지 못한다.**")
run.log("    영점을 기준으로 삼으면 그 차이가 곧 **창 내용의 특이성**이다(R34 ③)")

W1 = {}
for nm, D in (("LORO", D_LORO), *[(f"k{k}", D_PERS[k]) for k in K_PERS]):
    d = D[GATE_ARM] - D[ZERO_ARM]                     # ★ 짝지은 차(같은 레코드·같은 폴드)
    m_, lo_, hi_, n_ = boot_mean(d, SEED0 + 71)
    W1[nm] = dict(mean=m_, lo=lo_, hi=hi_, n=n_, mde=float(mde(lo_, hi_)),
                  a=float(np.nanmean(D[GATE_ARM])), b=float(np.nanmean(D[ZERO_ARM])))
    run.log(f"    {nm:<6} **{m_:+.4f}** [{lo_:+.4f}, {hi_:+.4f}] · n={n_} · "
            f"MDE {mde(lo_,hi_):.4f}   (성분 {GATE_ARM} {W1[nm]['a']:+.4f} · "
            f"{ZERO_ARM} {W1[nm]['b']:+.4f})")   # ★ R36 ⑤ — 성분 없이 인용 금지
d1 = W1["LORO"]
DIFF["W1"] = dict(**d1)
g_("W1", "⛔ 측정 불가" if d1["n"] < 3 else decide(d1["lo"], d1["hi"], 0.0, ">"),
   f"★★★ **주 대비** `{GATE_ARM} − {ZERO_ARM}` **{d1['mean']:+.4f}** "
   f"[{d1['lo']:+.4f}, {d1['hi']:+.4f}] · n={d1['n']}")

# 비교용 — Q7-S′ 의 옛 주 대비
S5 = {}
for nm, D in (("LORO", D_LORO), *[(f"k{k}", D_PERS[k]) for k in K_PERS]):
    m_, lo_, hi_, n_ = boot_mean(D["palign"] - D["pmorph"], SEED0 + 61)
    S5[nm] = dict(mean=m_, lo=lo_, hi=hi_, n=n_)
run.log(f"\n  (참고 · 옛 대비) `palign − pmorph` LORO {S5['LORO']['mean']:+.4f} "
        f"[{S5['LORO']['lo']:+.4f}, {S5['LORO']['hi']:+.4f}] — **보조로 강등**")

# ── ★ W4 — 영점 드리프트가 실재하나
run.log("\n  ★ W4 — 전 팔을 **자기 영점과 짝지어** 재보고 (k별 · R36 ④)")
run.log("    ▸ 외부 검토가 `shuf5` 의 k=20 드리프트(−0.0066)를 지적했다.")
run.log("      그게 `shuf5` **자기 MDE 안**이면 드리프트는 **확인된 게 아니다**")
run.log(f"    {'k':<6}{'shuf5':>10}{'shuf5 MDE':>12}{'MDE 밖?':>9}{'palign':>10}"
        f"{'palign-shuf5':>14}")
W4 = {}
for k in K_PERS:
    z = S1[f"{ZERO_ARM}@k{k}"]; g = S1[f"{GATE_ARM}@k{k}"]
    out = abs(z["mean"]) > z["mde"] if np.isfinite(z["mde"]) else False
    W4[f"k{k}"] = dict(zero=z["mean"], zero_mde=z["mde"], outside=bool(out),
                       gate=g["mean"], paired=W1[f"k{k}"]["mean"])
    run.log(f"    {k:<6}{z['mean']:>+10.4f}{z['mde']:>12.4f}{'예' if out else '아니오':>9}"
            f"{g['mean']:>+10.4f}{W1[f'k{k}']['mean']:>+14.4f}")
drift = any(v["outside"] for v in W4.values())
g_("W4", "❌ 기각" if drift else "✅ 지지",
   ("★ 영점이 자기 MDE 밖으로 움직인다 — k별 값을 **영점 짝지은 차로만** 읽어야 한다"
    if drift else
    "영점 드리프트는 **자기 잡음 안**이다 — 확인된 게 아니다. 짝지은 차는 그대로 유효"))
CONFIG["W1"] = W1; CONFIG["W4"] = W4; CONFIG["S5"] = S5
run.save_json("config", CONFIG)

In [ ]:
# CELL 7 — 【S3-E】 W3 추론 단위 민감도 · W2 필요 표본(등가 vs 우월)
run.log("\n" + "=" * 100)
run.log("【S3-E】 W3 추론 단위 · W2 필요 표본 — **등가 vs 우월**")
run.log("=" * 100)

# ── W3 — 레코드 부트가 과도하게 보수적인가
run.log("  W3 — 추론 단위 민감도 (프로브 `p_score`)")
run.log("    ⓐ 레코드평균 + 레코드부트  (현행 · R11)")
run.log("    ⓑ 쌍풀링   + 레코드부트  (쌍 수로 가중 · 군집 유지)")
run.log("    ⓒ 쌍풀링   + 쌍부트      (**군집 무시** — 추론에 쓰면 안 된다 · 비용 정량용)")
pw = PAIRW["p_score"]
if len(pw) < 3:
    g_("W3", "⛔ 측정 불가", "쌍이 선 레코드가 3 미만")
else:
    aucs = np.array([a for _, a, _ in pw], float)
    npr = np.array([n for _, _, n in pw], float)
    a_m, a_lo, a_hi, a_n = boot_mean(aucs, SEED0 + 101)
    rr = np.random.RandomState(SEED0 + 102)
    bb = []
    for _ in range(3000):
        j = rr.randint(0, len(aucs), len(aucs))
        bb.append(float((aucs[j] * npr[j]).sum() / npr[j].sum()))
    b_m = float((aucs * npr).sum() / npr.sum())
    b_lo, b_hi = float(np.percentile(bb, 2.5)), float(np.percentile(bb, 97.5))
    # ⓒ 쌍 단위 근사 — 풀링 AUROC 의 이항 SE(군집 무시). **추론용 아님**
    ntot = float(npr.sum())
    c_se = float(np.sqrt(max(b_m * (1 - b_m), 1e-12) / ntot))
    c_lo, c_hi = b_m - 1.96 * c_se, b_m + 1.96 * c_se
    W3 = dict(rec=dict(mean=a_m, lo=a_lo, hi=a_hi, mde=float(mde(a_lo, a_hi))),
              pool_rec=dict(mean=b_m, lo=b_lo, hi=b_hi, mde=float(mde(b_lo, b_hi))),
              pool_pair=dict(mean=b_m, lo=c_lo, hi=c_hi, mde=float(mde(c_lo, c_hi))),
              n_pair=int(ntot))
    for tag, d_ in (("ⓐ 레코드", W3["rec"]), ("ⓑ 쌍풀링·레코드부트", W3["pool_rec"]),
                    ("ⓒ 쌍풀링·쌍부트", W3["pool_pair"])):
        run.log(f"    {tag:<22} {d_['mean']:.4f} [{d_['lo']:.4f}, {d_['hi']:.4f}] · "
                f"MDE {d_['mde']:.4f}")
    ratio = W3["rec"]["mde"] / max(W3["pool_pair"]["mde"], 1e-12)
    run.log(f"    총 쌍 {int(ntot):,} · **군집 비용 = MDE {ratio:.1f}배**")
    same = abs(W3["rec"]["mde"] - W3["pool_rec"]["mde"]) <= 0.2 * W3["rec"]["mde"]
    g_("W3", "✅ 지지" if same else "⚠️ 미결",
       ("ⓐ≈ⓑ — **MDE 는 진짜다.** 쌍을 늘려도 안 되고 **레코드를 늘려야** 한다"
        if same else "ⓐ 와 ⓑ 가 갈린다 — 가중 방식이 결과를 바꾼다. 둘 다 보고한다"))
    run.log("    ▸ ⓒ 는 **군집을 무시**한 값이라 추론에 쓰면 안 된다. 레코드 간 이질성이")
    run.log("      얼마나 비싼지를 보이려고만 찍는다(R11)")
    CONFIG["W3"] = W3

# ── ★★ W2 — 필요 표본, 두 프레임
run.log("\n  ★★ W2 — 필요 표본: **등가**(Q7-S′ 가 쓴 식) vs **우월**(우리가 원하는 것)")
run.log("    ▸ 등가 = 「P 가 아무것도 안 준다」를 증명하는 데 필요한 수")
run.log("    ▸ 우월 = 「효과가 0 이 아니다」를 보이는 데 필요한 수 ← **이게 우리 질문이다**")
run.log(f"    {'관문':<8}{'점추정':>10}{'반폭':>9}{'등가':>10}{'우월50%':>10}{'우월80%':>10}"
        f"{'현재n':>7}")
TARGETS = [("S1@k5", S1[f"{GATE_ARM}@k5"], 0.0),
           ("S2", S2[GATE_ARM], 0.0),
           ("W1", W1["LORO"], 0.0),
           ("S5", S5["LORO"], 0.0)]
if "S3" in DIFF:
    TARGETS.append(("S3", DIFF["S3"], 0.5))
W2 = {}
for gname, d_, ref in TARGETS:
    val = d_.get("mean", d_.get("auc", float("nan")))
    eff = val - ref
    ne = need_equiv(d_["n"], d_["lo"], d_["hi"], eff, 0.01)
    n5 = need_super(d_["n"], d_["lo"], d_["hi"], eff, False)
    n8 = need_super(d_["n"], d_["lo"], d_["hi"], eff, True)
    # S3 는 **매칭 가능 레코드**가 단위다 — 총 레코드로 환산해서 같이 찍는다
    scale = (len(REC_OK) / max(d_["n"], 1)) if gname == "S3" else 1.0
    W2[gname] = dict(effect=float(eff), half=float(mde(d_["lo"], d_["hi"])),
                     equiv=(None if ne is None else float(ne)),
                     sup50=float(n5), sup80=float(n8), n=int(d_["n"]),
                     sup50_total=float(n5 * scale), sup80_total=float(n8 * scale))
    fe = "도달불가" if ne is None else f"{ne:.0f}"
    run.log(f"    {gname:<8}{eff:>+10.4f}{mde(d_['lo'],d_['hi']):>9.4f}{fe:>10}"
            f"{n5:>10.0f}{n8:>10.0f}{d_['n']:>7}")
    if gname == "S3":
        run.log(f"      └ S3 단위는 **매칭 가능 레코드**다 → 총 레코드 환산 "
                f"우월50% **{n5*scale:.0f}** · 우월80% **{n8*scale:.0f}** "
                f"(매칭 성립률 {d_['n']}/{len(REC_OK)})")
run.log(f"\n    도달 가능 풀 — 현재 **{POOL_NOW}**(SVDB78+MITBIH48) · 풀링 상한 ~{POOL_MAX}")
run.log("    ★ 프레임을 바꾸면 **가장 비싼 관문과 가장 싼 관문이 뒤바뀐다** —")
run.log("      등가에서 S3 는 「도달 불가」였지만 우월에서는 **가장 싸다**")
run.log("    ⚠️ 그래도 80% 검정력 기준으로는 **풀링 상한도 못 넘을 수 있다** —")
run.log("      바뀌는 건 「불가능」→「비싸다」이지 「도달권」이 아니다. 위 숫자로 판단한다")
CONFIG["W2"] = W2
run.save_json("config", CONFIG)

In [ ]:
# CELL 8 — 【S3-F】 ★★★ W0 재현 증명 (여기서 깨지면 위 전부 무효)
run.log("\n" + "=" * 100)
run.log("【S3-F】 W0 — Q7-S′ 공식 실행 `20260804T0658` 재현 증명 (R35 ⑦)")
run.log("=" * 100)
GOT = {
    "S2.palign": S2["palign"]["mean"], "S2.pmorph": S2["pmorph"]["mean"],
    "S2.noise5": S2["noise5"]["mean"], "S2.shuf5":  S2["shuf5"]["mean"],
    **{f"S1.{GATE_ARM}@k{k}": S1[f"{GATE_ARM}@k{k}"]["mean"] for k in K_PERS},
    "S3.p_score": S3["p_score"]["auc"], "S3.pr_q8": S3["pr_q8"]["auc"],
    "S3.p_miss":  S3["p_miss"]["auc"],
    "S5.LORO": S5["LORO"]["mean"],
}
run.log(f"  {'항목':<18}{'Q7-S′':>10}{'재현':>10}{'Δ':>10}  판정")
bad, warn, skip = [], [], []
for k_, ref_ in REF.items():
    if k_ not in GOT:                     # 설정을 줄여 돌린 경우 — 조용히 넘기지 않고 찍는다
        skip.append(k_); continue
    g = float(GOT[k_])
    d = abs(g - ref_)
    tag = "✅" if d <= TOL_WARN else ("⚠️" if d <= TOL_FAIL else "❌")
    if d > TOL_FAIL: bad.append((k_, ref_, g, d))
    elif d > TOL_WARN: warn.append((k_, ref_, g, d))
    run.log(f"  {k_:<18}{ref_:>10.4f}{g:>10.4f}{g-ref_:>+10.4f}  {tag}")
if skip:
    run.log(f"  ⚠️ 기준에는 있으나 이번 실행에 없는 항목 {skip} — 설정이 다르다")
run.log(f"\n  판정 레코드 — Q7-S′ S2 {REF_N.get('S2', '?')} / 재현 {S2[GATE_ARM]['n']} · "
        f"S3 {REF_N.get('S3', '?')} / 재현 {S3['p_score']['n']}")
if REF_N and (S2[GATE_ARM]["n"] != REF_N.get("S2") or
              S3["p_score"]["n"] != REF_N.get("S3")):
    bad.append(("판정 레코드 수", float(REF_N.get("S2", 0)),
                float(S2[GATE_ARM]["n"]), float("nan")))
CONFIG["W0"] = dict(ref=REF, got={k_: float(v) for k_, v in GOT.items()},
                    n_bad=len(bad), n_warn=len(warn), skipped=skip)
run.save_json("config", CONFIG)
if bad:
    g_("W0", "❌ 기각", f"**{len(bad)}개 항목이 {TOL_FAIL} 를 넘는다**")
    for k_, r_, g_v, d_ in bad:
        run.log(f"    ✗ {k_}: 기준 {r_:.4f} vs 재현 {g_v:.4f}")
    raise AssetError(
        "재현 증명 실패 — Q7-S′ 값을 다시 내지 못했다. 자산(svdb_pdelin.npz)이나 "
        "라이브러리 버전이 바뀐 것이고, 그러면 W1~W5 를 Q7-S′ 와 나란히 놓을 근거가 "
        f"없다. 어긋난 항목 {[b[0] for b in bad]}")
g_("W0", "✅ 지지" if not warn else "⚠️ 미결",
   f"재현 완료 — 전 항목 |Δ| ≤ {TOL_FAIL}"
   + (f" (단 {len(warn)}개가 {TOL_WARN} 초과: {[w[0] for w in warn]})" if warn else ""))
run.log("  ▸ 로그가 소수점 4자리로 찍혔으므로 |Δ| ≤ 5e-5 는 **반올림 오차**다")

In [ ]:
# CELL 9 — 【S3-G】 그림 · 요약 · 마무리
# ⚠️ Colab 기본 matplotlib 에 **한글이 없어** 축·범례는 ASCII 로만 쓴다.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.6))

# ① 주 대비 비교 — 옛(palign-pmorph) vs 새(palign-shuf5) + 성분
lbl = ["palign", "pmorph", "shuf5", "palign-pmorph", "palign-shuf5"]
val = [S2["palign"]["mean"], S2["pmorph"]["mean"], S2["shuf5"]["mean"],
       S5["LORO"]["mean"], W1["LORO"]["mean"]]
elo = [S2["palign"]["lo"], S2["pmorph"]["lo"], S2["shuf5"]["lo"],
       S5["LORO"]["lo"], W1["LORO"]["lo"]]
ehi = [S2["palign"]["hi"], S2["pmorph"]["hi"], S2["shuf5"]["hi"],
       S5["LORO"]["hi"], W1["LORO"]["hi"]]
cols = ["tab:blue"] * 3 + ["tab:gray", "tab:red"]
ax[0].errorbar(val, np.arange(len(lbl)),
               xerr=[np.array(val) - np.array(elo), np.array(ehi) - np.array(val)],
               fmt="o", capsize=4, ecolor="k", ls="none")
for i, c in enumerate(cols):
    ax[0].scatter([val[i]], [i], color=c, zorder=3)
ax[0].axvline(0, color="k", lw=.9)
ax[0].set_yticks(range(len(lbl))); ax[0].set_yticklabels(lbl, fontsize=8)
ax[0].set_xlabel("W1 : dAUPRC, components and contrasts")
ax[0].grid(alpha=.3, axis="x")

# ② 매칭 폭 곡선 — f1 항등 대조가 핵심
ws = list(MATCH_W)
ax[1].plot(ws, [W5[w]["f1_auc"] for w in ws], "o-", color="tab:red",
           label="f1 (identity control)")
ax[1].plot(ws, [W5[w]["ps_auc"] for w in ws], "s-", color="tab:blue", label="p_score")
ax[1].axhline(0.5, ls="--", color="k", lw=.9)
ax[1].axhspan(0.5 - IDENTITY_TOL, 0.5 + IDENTITY_TOL, color="tab:green", alpha=.12)
for w in ws:
    ax[1].annotate(f"n={W5[w]['n_rec']}", (w, W5[w]["f1_auc"]), fontsize=6,
                   textcoords="offset points", xytext=(0, -11), ha="center")
ax[1].set_xscale("log", base=2); ax[1].set_xticks(ws)
ax[1].set_xticklabels([str(w) for w in ws])
ax[1].set_xlabel("matching width (samples @360Hz)")
ax[1].set_ylabel("matched AUROC")
ax[1].legend(fontsize=7); ax[1].grid(alpha=.3)

# ③ 필요 표본 — 등가 vs 우월
gn = [g for g, _, _ in TARGETS]
xs = np.arange(len(gn))
eq = [W2[g]["equiv"] if W2[g]["equiv"] is not None else np.nan for g in gn]
s5v = [W2[g]["sup50"] for g in gn]; s8v = [W2[g]["sup80"] for g in gn]
ax[2].bar(xs - 0.26, eq, 0.24, label="equivalence (old)", color="tab:gray")
ax[2].bar(xs, s5v, 0.24, label="superiority 50%", color="tab:orange")
ax[2].bar(xs + 0.26, s8v, 0.24, label="superiority 80%", color="tab:red")
ax[2].axhline(POOL_NOW, ls="--", color="k", lw=.9)
ax[2].axhline(POOL_MAX, ls=":", color="k", lw=.9)
ax[2].set_yscale("log"); ax[2].set_xticks(xs); ax[2].set_xticklabels(gn)
ax[2].set_ylabel("required records (log)")
ax[2].set_xlabel("gate  (dashed = reachable pool 126 / 201)")
ax[2].legend(fontsize=7); ax[2].grid(alpha=.3, axis="y")
for i, g in enumerate(gn):
    if W2[g]["equiv"] is None:
        ax[2].annotate("unreachable", (i - 0.26, 1.5), fontsize=6, rotation=90, ha="center")
fig.tight_layout()
PNG = run.save_fig("q7s3_recompute", fig)
plt.close(fig); display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
no_ = lambda k: VERD.get(k, "").startswith("❌")
un_ = lambda k: VERD.get(k, "").startswith("⛔")
for g in ("W0", "W5", "W1", "W3", "W4", "S3"):
    run.log(f"  {g:<4}{VERD.get(g, '(미실행)')}")
run.log("")
if not ok_("W0"):
    run.log("  ⛔ 재현이 안 됐다 — 아래 어떤 문장도 Q7-S′ 와 나란히 놓지 않는다")
elif no_("W5"):
    run.log("  ⛔⛔ **매칭이 샌다 — S3 를 철회한다.** `f1` 항등 대조가 0.5 를 뗐다는 건")
    run.log("      「정확 매칭 층 안에서는 조기성이 상수」라는 S3 의 전제가 거짓이라는 뜻이다.")
    run.log("      Q7-S′ 의 S3 +0.0272 는 잔여 조기성을 다시 잰 것일 수 있다(R24).")
    run.log("      → **더 좁은 매칭**(폭 곡선의 누출 0 구간)에서만 다시 읽는다")
elif ok_("W1"):
    run.log("  ★★ **창 내용이 특이적이다** — `palign` 이 자기 영점을 유의하게 넘는다.")
    run.log("     S3 의 초과분을 액면대로 읽어도 되고, 다음은 **크기**의 문제다.")
    run.log("     → Q7-V(자가 병목인가) → 그다음 코호트")
else:
    run.log("  ⚠️ 주 대비 미결 — 부호와 **상한**을 함께 쓴다(R36 ①).")
    run.log(f"     `palign − shuf5` 상한 **{W1['LORO']['hi']:+.4f}** — 있어도 이보다 크지 않다")
    run.log("     → 크기 문제인지 자 문제인지는 **Q7-V** 가 가른다")
run.log("\n  ★ 프레임 정정 (R30 ① · R36 ②)")
if "S3" in W2:
    run.log(f"    S3 — 등가 「도달 불가」 → **우월 50% {W2['S3']['sup50_total']:.0f}** · "
            f"80% {W2['S3']['sup80_total']:.0f} (총 레코드 환산)")
run.log(f"    풀 {POOL_NOW}(현재) / ~{POOL_MAX}(풀링) 과 비교해 **각 관문이 도달권인지**를 위 표로 판단")
run.log("\n  ⚠️ **이 런은 새 데이터를 쓰지 않았다** — 같은 자산·같은 시드·같은 코드다.")
run.log("     바뀐 것은 ① 필요 표본의 **프레임** ② 주 대비의 **기준**(pmorph→shuf5)")
run.log("     ③ 프로브 선택을 **LORO 로**(Q7-S′ 는 max — 자백한 편의) ④ **`f1` 항등 대조** 추가")

run.finish({
    "exp_id": "quest46_q7s3_recompute",
    "metric": "palign_minus_zero_dAUPRC",
    "value": float(W1["LORO"]["mean"]),
    "passed": bool(ok_("W0") and ok_("W5") and ok_("W1")),
    "summary": ("Q7-S′ 재계산 — 필요 표본을 등가에서 우월 프레임으로, 주 대비를 "
                "palign-pmorph 에서 palign-shuf5(영점 기준)로, 프로브 선택을 LORO 로, "
                "그리고 f1 항등 대조로 정확 매칭이 정말 정확한지 확인."),
    "verdicts": VERD, "diffs": DIFF, "rule_check": RULE_CHECK,
    "W0": CONFIG.get("W0", {}), "W1": CONFIG.get("W1", {}), "W2": CONFIG.get("W2", {}),
    "W3": CONFIG.get("W3", {}), "W4": CONFIG.get("W4", {}), "W5": CONFIG.get("W5", {}),
    "S1": CONFIG.get("S1", {}), "S2": CONFIG.get("S2", {}), "S3": CONFIG.get("S3", {}),
    "S5": S5, "asset": CONFIG.get("asset", {}), "feat": CONFIG.get("feat", {}), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `ingest_run.py --quest ailab-2026-0046 --step recompute-frames`")